# Carry and roll-down inventory tilt (EDGE-E6)

Does tilting a Treasury inventory towards the tenors with the most carry and roll-down per unit DV01 (weights 1 + 0.5 z, no shorts, same total DV01) beat an equal-DV01 inventory over the next month, net of funding at the 1-month bill plus a spread? Pre-registration `research/prereg/E6.yaml`; fitting 2015 to 2020, holdout rebalances 2021-01 to 2026-05. Constant-maturity par yields, no bid-offer.

Every number below is recomputed from the committed fixtures and checked against the ledger row; the plain-language finding is in `docs/research/EDGE_STUDIES.md`.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from ratesresearch import backtest_analysis as ba
from ratesresearch.edge import report

ROOT = Path.cwd()
while not (ROOT / "data" / "presets.yaml").is_file():
    ROOT = ROOT.parent


def show(df: pd.DataFrame, digits: int = 4) -> None:
    display(Markdown(ba.to_markdown(df.round(digits))))

In [ ]:
from ratesresearch.edge import e6_carry

res = e6_carry.run(ROOT)
row = report.ledger_row(ROOT, "EDGE-E6")
display(Markdown(report.check_against_ledger(res, row)))
show(report.tests_table(res, row))

Band points and periods (unadjusted p; only the central holdout row is a registered test).

In [ ]:
show(report.band_table(res, "diff"))

In [ ]:
show(
    pd.DataFrame(
        [
            {
                "band": b,
                **{
                    p: x[p]["tilt_mean_bp"] - x[p]["baseline_mean_bp"]
                    for p in ("fitting", "holdout")
                },
            }
            for b, x in res["band"].items()
        ]
    )
)

In [ ]:
show(res["monthly"][["month", "baseline", "tilt", "diff"]].tail(12))